# Calculation of Statistical Metrics #

In [2]:
import os
import math
import itertools
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256   # 🔥 reduce from 512 → faster + less memory

In [3]:
# =========================
# LOAD MODEL
# =========================
model_name = "microsoft/codebert-base-mlm"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name).to(DEVICE)
model.eval()

# =========================
# LOAD DATA (FIXED)
# =========================
def load_data(folder):
    texts, labels = [], []

    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)

        if not os.path.exists(subfolder):
            raise ValueError(f"Missing folder: {subfolder}")

        for file in os.listdir(subfolder):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)

    return texts, np.array(labels)

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
# =========================
# FAST + MEMORY SAFE METRICS
# =========================
def compute_metrics(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    )

    input_ids = inputs["input_ids"].to(DEVICE)
    seq_len = input_ids.size(1)

    logits_list = []
    target_ids = []

    for i in range(1, seq_len - 1):
        masked = input_ids.clone()
        masked[0, i] = tokenizer.mask_token_id

        with torch.no_grad():
            logits = model(masked).logits[0, i]

        # 🔥 move immediately to CPU
        logits_list.append(logits.cpu())
        target_ids.append(input_ids[0, i].cpu())

    logits_tensor = torch.stack(logits_list)  # CPU
    targets = torch.tensor(target_ids)

    log_probs = torch.log_softmax(logits_tensor, dim=-1)
    probs = torch.softmax(logits_tensor, dim=-1)

    # (1) Log Likelihood
    # ll = log_probs[torch.arange(len(targets)), targets].sum().item()

    # (2) Log Rank
    sorted_ids = torch.argsort(logits_tensor, dim=-1, descending=True)
    ranks = (sorted_ids == targets.unsqueeze(1)).nonzero()[:, 1] + 1
    log_rank = torch.log(ranks.float()).mean().item()

    # (3) Entropy
    entropy = (-(probs * log_probs).sum(dim=-1)).mean().item()

    # (4–6) GLTR
    t = targets.unsqueeze(1)

    top10 = torch.topk(logits_tensor, 10, dim=-1).indices
    top100 = torch.topk(logits_tensor, 100, dim=-1).indices
    top1000 = torch.topk(logits_tensor, 1000, dim=-1).indices

    top10_ratio = (top10 == t).any(dim=1).float().mean().item()
    top100_ratio = (top100 == t).any(dim=1).float().mean().item()
    top1000_ratio = (top1000 == t).any(dim=1).float().mean().item()

    # free GPU memory
    torch.cuda.empty_cache()

    return [log_rank, entropy, top10_ratio, top100_ratio, top1000_ratio]



# =========================
# DATASET METRICS + PROGRESS
# =========================
def compute_dataset(texts, name):
    results = []
    for text in tqdm(texts, desc=f"Processing {name}"):
        results.append(compute_metrics(text))
    return np.array(results)


# =========================
# MLP CLASSIFIER
# =========================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.net(x)


def train_and_eval(X_train, y_train, X_test, y_test):
    model = MLP(X_train.shape[1]).to(DEVICE)

    X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    y_train = torch.tensor(y_train).to(DEVICE)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(50):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        preds = model(X_test).argmax(dim=1).cpu().numpy()

    return accuracy_score(y_test, preds)

In [8]:
print("\nUsing ALL metrics (except LL)...")

X_train = train_metrics
X_test = test_metrics

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

acc = train_and_eval(X_train, train_labels, X_test, test_labels)

print("\n======================")
print("FINAL ACCURACY:", acc)


Using ALL metrics (except LL)...

FINAL ACCURACY: 0.5099800399201597


In [24]:
print("Train Metrics Shape:", train_metrics.shape)

Train Metrics Shape: (6190, 5)


#COMBINING WITH CodeBERT

In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

In [7]:
data = np.load("metrics_data.npz")

train_metrics = data["train_metrics"]
train_labels = data["train_labels"]
test_metrics = data["test_metrics"]
test_labels = data["test_labels"]

In [8]:
def load_data(folder):
    texts, labels = [], []

    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)

        for file in os.listdir(subfolder):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)

    return texts, np.array(labels)


train_texts, _ = load_data("../Text_Files/Train")
test_texts, _ = load_data("../Text_Files/Test_0")

In [5]:
scaler = StandardScaler()
train_metrics = scaler.fit_transform(train_metrics)
test_metrics = scaler.transform(test_metrics)

In [11]:
train_metrics = train_metrics[:, 1:]
test_metrics = test_metrics[:, 1:]

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
codebert = AutoModel.from_pretrained(model_name).to(device)

In [10]:
class HybridDataset(Dataset):
    def __init__(self, texts, metrics, labels):
        self.texts = texts
        self.metrics = torch.tensor(metrics, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "metrics": self.metrics[idx],
            "label": self.labels[idx]
        }

In [12]:
train_dataset = HybridDataset(train_texts, train_metrics, train_labels)
test_dataset = HybridDataset(test_texts, test_metrics, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [13]:
class HybridModel(nn.Module):
    def __init__(self, codebert):
        super().__init__()
        self.codebert = codebert

        self.norm = nn.LayerNorm(768 + 5)   # 🔥 ADD THIS

        self.classifier = nn.Sequential(
            nn.Linear(768 + 5, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask, metrics):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        combined = torch.cat([cls_embedding, metrics], dim=1)

        combined = self.norm(combined)   # 🔥 IMPORTANT

        return self.classifier(combined)

In [14]:
model = HybridModel(codebert).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [51]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        metrics = batch["metrics"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask, metrics)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1: 100%|██████████| 774/774 [01:16<00:00, 10.14it/s]


Epoch 1 Loss: 305.9305


Epoch 2: 100%|██████████| 774/774 [01:16<00:00, 10.08it/s]


Epoch 2 Loss: 222.6371


Epoch 3: 100%|██████████| 774/774 [01:16<00:00, 10.06it/s]

Epoch 3 Loss: 170.7634


In [53]:
# Save checkpoint
torch.save(model.state_dict(), "../checkpoints/stat/hybrid_codebert.pt")

In [52]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        metrics = batch["metrics"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask, metrics)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Test Accuracy:", correct / total)

Test Accuracy: 0.7554890219560878


In [ ]:
#0.7574 as Test Accuracy with 3 Epochs

In [54]:
def load_data(folder):
    texts, labels = [], []

    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)

        for file in os.listdir(subfolder):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)

    return texts, np.array(labels)


train_texts, train_labels = load_data("../Text_Files/Train")
test_texts, test_labels = load_data("../Text_Files/Test_0")

In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
codebert = AutoModel.from_pretrained(model_name).to(device)

In [56]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": self.labels[idx]
        }

In [57]:
train_dataset = TextDataset(train_texts, train_labels)
test_dataset = TextDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [58]:
class CodeBERTClassifier(nn.Module):
    def __init__(self, codebert):
        super().__init__()
        self.codebert = codebert

        self.norm = nn.LayerNorm(768)

        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        cls_embedding = self.norm(cls_embedding)  # normalization

        return self.classifier(cls_embedding)

In [59]:
model = CodeBERTClassifier(codebert).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [60]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1: 100%|██████████| 774/774 [01:15<00:00, 10.19it/s]


Epoch 1 Loss: 312.5540


Epoch 2: 100%|██████████| 774/774 [01:16<00:00, 10.08it/s]


Epoch 2 Loss: 203.4942


Epoch 3: 100%|██████████| 774/774 [01:16<00:00, 10.06it/s]

Epoch 3 Loss: 151.5446


In [15]:
def evaluate(folder_path):
    texts, labels = load_data(folder_path)

    dataset = TextDataset(texts, labels)
    loader = DataLoader(dataset, batch_size=8)

    correct, total = 0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            preds = outputs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total if total > 0 else 0

In [62]:
# =========================
# RUN OVER ALL TEST SETS
# =========================
results = {}

for i in range(10):
    folder = f"../Text_Files/Test_{i}"

    if not os.path.exists(folder):
        print(f"{folder} not found, skipping...")
        continue

    acc = evaluate(folder)
    results[folder] = acc

    print(f"{folder} -> Accuracy: {acc:.4f}")


# =========================
# SUMMARY
# =========================
print("\n===== SUMMARY =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

print("\nAverage Accuracy:", sum(results.values()) / len(results))

../Text_Files/Test_0 -> Accuracy: 0.7405
../Text_Files/Test_1 -> Accuracy: 0.7605
../Text_Files/Test_2 -> Accuracy: 0.7478
../Text_Files/Test_3 -> Accuracy: 0.7335
../Text_Files/Test_4 -> Accuracy: 0.7335
../Text_Files/Test_5 -> Accuracy: 0.7894
../Text_Files/Test_6 -> Accuracy: 0.7186
../Text_Files/Test_7 -> Accuracy: 0.7156
../Text_Files/Test_8 -> Accuracy: 0.7236
../Text_Files/Test_9 -> Accuracy: 0.6976

===== SUMMARY =====
../Text_Files/Test_0: 0.7405
../Text_Files/Test_1: 0.7605
../Text_Files/Test_2: 0.7478
../Text_Files/Test_3: 0.7335
../Text_Files/Test_4: 0.7335
../Text_Files/Test_5: 0.7894
../Text_Files/Test_6: 0.7186
../Text_Files/Test_7: 0.7156
../Text_Files/Test_8: 0.7236
../Text_Files/Test_9: 0.6976

Average Accuracy: 0.7360557702329331


In [63]:
torch.save(model.state_dict(), "../checkpoints/stat/only_codebert.pt")

In [16]:
# Load later
model.load_state_dict(torch.load("../checkpoints/stat/hybrid_codebert.pt"))
model.eval()

HybridModel(
  (codebert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [17]:
def evaluate(folder_path, metrics):
    texts, labels = load_data(folder_path)

    dataset = HybridDataset(texts, metrics, labels)
    loader = DataLoader(dataset, batch_size=8)

    correct, total = 0, 0

    model.eval()

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            metrics = batch["metrics"].to(device)
            y = batch["label"].to(device)

            outputs = model(input_ids, attention_mask, metrics)
            preds = outputs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total if total > 0 else 0

results = {}

for i in range(10):
    folder = f"../Text_Files/Test_{i}"

    if not os.path.exists(folder):
        print(f"{folder} not found, skipping...")
        continue

    # ⚠️ IMPORTANT: load corresponding metrics
    data = np.load(f"metrics_test_{i}.npz")
    test_metrics = data["test_metrics"]

    acc = evaluate(folder, test_metrics)
    results[folder] = acc

    print(f"{folder} -> Accuracy: {acc:.4f}")


print("\nAverage Accuracy:", sum(results.values()) / len(results))

../Text_Files/Test_0 -> Accuracy: 0.7565
../Text_Files/Test_1 -> Accuracy: 0.7395
../Text_Files/Test_2 -> Accuracy: 0.7123
../Text_Files/Test_3 -> Accuracy: 0.7106
../Text_Files/Test_4 -> Accuracy: 0.7016
../Text_Files/Test_5 -> Accuracy: 0.7934
../Text_Files/Test_6 -> Accuracy: 0.6866
../Text_Files/Test_7 -> Accuracy: 0.6577
../Text_Files/Test_8 -> Accuracy: 0.6866
../Text_Files/Test_9 -> Accuracy: 0.6577

Average Accuracy: 0.7102534831814203


In [76]:
import os
import numpy as np
from tqdm import tqdm

# =========================
# Reuse your existing:
# - load_data()
# - compute_dataset()
# =========================

for i in range(10):
    folder = f"../Text_Files/Test_{i}"

    if not os.path.exists(folder):
        print(f"{folder} not found, skipping...")
        continue

    print(f"\nProcessing {folder}...")

    texts, labels = load_data(folder)

    # 🔥 Compute metrics (uses your optimized function)
    test_metrics = compute_dataset(texts, f"Test_{i}")

    # 🔥 Save
    save_path = f"metrics_test_{i}.npz"
    np.savez(save_path,
             test_metrics=test_metrics,
             test_labels=labels)

    print(f"Saved → {save_path}")


Processing ../Text_Files/Test_0...


Processing Test_0: 100%|██████████| 1002/1002 [22:20<00:00,  1.34s/it]


Saved → metrics_test_0.npz

Processing ../Text_Files/Test_1...


Processing Test_1: 100%|██████████| 1002/1002 [22:17<00:00,  1.34s/it]


Saved → metrics_test_1.npz

Processing ../Text_Files/Test_2...


...

Saved → metrics_test_7.npz

Processing ../Text_Files/Test_8...


Processing Test_8: 100%|██████████| 1002/1002 [23:51<00:00,  1.43s/it]


Saved → metrics_test_8.npz

Processing ../Text_Files/Test_9...


Processing Test_9: 100%|██████████| 1002/1002 [21:31<00:00,  1.29s/it]

Saved → metrics_test_9.npz


In [18]:
data = np.load(f"metrics_test_{i}.npz")
test_metrics = data["test_metrics"]

print(test_metrics.shape)

(1002, 5)
